In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-addon-aqc-tensor qiskit-addon-utils scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pagsisimula sa Approximate quantum compilation gamit ang tensor networks (AQC-Tensor)


<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na requirements.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
qiskit-addon-utils~=0.3.0
qiskit-addon-aqc-tensor[aer,quimb-jax]~=0.2.0; sys.platform != 'darwin'
scipy~=1.16.3
```
</details>
Ipinapakita ng gabay na ito ang isang simpleng halimbawa para makapagsimula sa AQC-Tensor. Sa halimbawang ito, kukuha ka ng Trotter circuit na nagsi-simulate ng ebolusyon ng isang transverse field Ising model at gagamitin ang pamamaraang AQC-Tensor para bawasan ang lalim ng circuit. Bukod dito, kailangan din ng halimbawang ito ang `qiskit-addon-utils` package para sa problem generator, `qiskit-aer` para sa tensor-network simulation, at `scipy` para sa parameter optimization.

Para magsimula, alalahanin na ang Hamiltonian ng transverse field Ising model ay may sumusunod na anyo

$$ \mathcal{H}_{Ising} = \sum_{i=1}^N J_{i,(i+1)}Z_iZ_{i+1} + h_i X_i $$

kung saan ipinapalagay natin ang periodic boundary conditions na nangangahulugang para sa $i=10$ makukuha natin ang $i+1=11\rightarrow 1$, ang $J$ ay ang coupling strength sa pagitan ng dalawang site, at ang $h$ ay ang lakas ng external na magnetic field.

Ang sumusunod na code snippet ay gagawa ng Hamiltonian ng isang 10-site Ising chain na may periodic boundary conditions.

In [1]:
from qiskit.transpiler import CouplingMap
from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit.synthesis import SuzukiTrotter
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_aqc_tensor import generate_ansatz_from_circuit
from qiskit_addon_aqc_tensor.simulation import tensornetwork_from_circuit
from qiskit_addon_aqc_tensor.simulation import compute_overlap
from qiskit_addon_aqc_tensor.objective import MaximizeStateFidelity
from qiskit_aer import AerSimulator
from scipy.optimize import OptimizeResult, minimize


# Generate some coupling map to use for this example
coupling_map = CouplingMap.from_heavy_hex(3, bidirectional=False)

# Choose a 10-qubit circle on this coupling map
reduced_coupling_map = coupling_map.reduce(
    [0, 13, 1, 14, 10, 16, 4, 15, 3, 9]
)

# Get a qubit operator describing the Ising field model
hamiltonian = generate_xyz_hamiltonian(
    reduced_coupling_map,
    coupling_constants=(0.0, 0.0, 1.0),
    ext_magnetic_field=(0.4, 0.0, 0.0),
)

## Hatiin ang time evolution sa dalawang bahagi para sa classical at quantum na pagpapatupad
Ang pangkalahatang layunin ng halimbawang ito ay i-simulate ang time evolution ng model Hamiltonian. Ginagawa natin ito dito sa pamamagitan ng Trotter evolution, na hahatiin sa dalawang bahagi.

1. Isang paunang bahagi na masi-simulate gamit ang matrix product states (MPS). Ito ang magiging 'kompil' gamit ang AQC-Tensor.
2. Isang kasunod na bahagi na ipapatupad sa quantum hardware.

Pag-eevolvein natin ang sistema hanggang sa oras na $t_f=5$ at gagamitin ang AQC-Tensor para i-compress ang time evolution hanggang sa oras na $t=4$, pagkatapos ay mag-eevolve gamit ang ordinaryong Trotter steps hanggang $t_f$.

Mula dito, gagawa tayo ng dalawang circuit — isa ang ico-compress gamit ang AQC-Tensor at isa ang ipapatupad sa QPU. Para sa unang circuit, dahil ito ay klasikal na si-simulate gamit ang matrix product states, gagamit tayo ng maraming layers para mabawasan ang Trotter error. Samantala, ang pangalawang circuit na nag-si-simulate ng time evolution mula $t_i=4$ hanggang $t_f=5$ ay gagamit ng mas kaunting layers para mabawasan ang lalim.

In [2]:
# Generate circuit to be compressed
aqc_evolution_time = 4.0
aqc_target_num_trotter_steps = 45

aqc_target_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=aqc_target_num_trotter_steps),
    time=aqc_evolution_time,
)

# Generate circuit to execute on hardware
subsequent_evolution_time = 1.0
subsequent_num_trotter_steps = 5

subsequent_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=subsequent_num_trotter_steps),
    time=subsequent_evolution_time,
)

Para sa paghahambing, gagawa rin tayo ng ikatlong circuit — isa na nag-eevolve hanggang $t=4$, ngunit may parehong bilang ng layers tulad ng pangalawang circuit na nag-eevolve mula $t_i=4$ hanggang $t_f=5$. Ito ang circuit na *sana ay* ginamit natin kung hindi tayo gumamit ng pamamaraang AQC-Tensor. Tatawagin natin itong *comparison* circuit.

In [3]:
aqc_comparison_num_trotter_steps = int(
    subsequent_num_trotter_steps
    / subsequent_evolution_time
    * aqc_evolution_time
)
aqc_comparison_num_trotter_steps

comparison_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=aqc_comparison_num_trotter_steps),
    time=aqc_evolution_time,
)

## Gumawa ng ansatz at buuin ang MPS simulation
Susunod, gagawa tayo ng ansatz na io-optimize natin. Mag-eevolve ito sa parehong evolution time tulad ng ating unang circuit (mula $t_i=0$ hanggang $t_f=4$), ngunit may mas kaunting Trotter steps.

Kapag nagawa na ang circuit, ipapasa natin ito sa `generate_ansatz_from_circuit()` function ng AQC-Tensor, na sinusuri ang two-qubit connectivity at nagbabalik ng dalawang bagay. Una ay isang nabuong ansatz circuit na may parehong two-qubit connectivity, at pangalawa ay isang set ng mga parameter na, kapag nilagyan ng halaga ang ansatz, nagbibigay ng input circuit.

In [4]:
aqc_ansatz_num_trotter_steps = 5

aqc_good_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=aqc_ansatz_num_trotter_steps),
    time=aqc_evolution_time,
)

aqc_ansatz, aqc_initial_parameters = generate_ansatz_from_circuit(
    aqc_good_circuit, qubits_initially_zero=True
)
aqc_ansatz.draw("mpl", fold=-1)

<Image src="../docs/images/guides/qiskit-addons-aqc-get-started/extracted-outputs/e08edb92-da1f-4131-85f5-f89006f7a2dd-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-aqc-get-started/extracted-outputs/e08edb92-da1f-4131-85f5-f89006f7a2dd-0.svg)

Susunod, itatayo natin ang MPS representation ng estado na aaproksimahin ng AQC. Kalkulahin din natin ang fidelity $|\langle\psi_1 | \psi_2 \rangle |^2$ sa pagitan ng estado na ginawa ng comparison circuit kumpara sa circuit na gumagawa ng target state (na gumamit ng mas maraming Trotter steps).

In [5]:
# Generate MPS simulator settings and obtain the MPS representation of the target state
simulator_settings = AerSimulator(
    method="matrix_product_state",
    matrix_product_state_max_bond_dimension=100,
)
aqc_target_mps = tensornetwork_from_circuit(
    aqc_target_circuit, simulator_settings
)


# Compute the fidelity between the MPS representation of the target state and the state produced by the comparison circuit
comparison_mps = tensornetwork_from_circuit(
    comparison_circuit, simulator_settings
)
comparison_fidelity = (
    abs(compute_overlap(comparison_mps, aqc_target_mps)) ** 2
)
print(f"Comparison fidelity: {comparison_fidelity}")

Comparison fidelity: 0.9997111919739693


## Optimize the parameters of the ansatz using the MPS

Lastly, we will optimize the ansatz circuit such that it produces the target state better than our `comparison_fidelity`. The cost function to minimize will be the `MaximizeStateFidelity` and will be optimized using scipy's L-BFGS optimizer.

In [6]:
objective = MaximizeStateFidelity(
    aqc_target_mps, aqc_ansatz, simulator_settings
)

stopping_point = 1 - comparison_fidelity


def callback(intermediate_result: OptimizeResult):
    print(f"Intermediate result: Fidelity {1 - intermediate_result.fun:.8}")
    if intermediate_result.fun < stopping_point:
        # Good enough for now
        raise StopIteration


result = minimize(
    objective,
    aqc_initial_parameters,
    method="L-BFGS-B",
    jac=True,
    options={"maxiter": 100},
    callback=callback,
)
if (
    result.status
    not in (
        0,
        1,
        99,
    )
):  # 0 => success; 1 => max iterations reached; 99 => early termination via StopIteration
    raise RuntimeError(
        f"Optimization failed: {result.message} (status={result.status})"
    )

print(f"Done after {result.nit} iterations.")
aqc_final_parameters = result.x

Intermediate result: Fidelity 0.95084365


Intermediate result: Fidelity 0.98409893


Intermediate result: Fidelity 0.99142033


Intermediate result: Fidelity 0.99521405


Intermediate result: Fidelity 0.99566673


...

Intermediate result: Fidelity 0.99956469


Intermediate result: Fidelity 0.99964488


Intermediate result: Fidelity 0.99967419


Intermediate result: Fidelity 0.99968821


Intermediate result: Fidelity 0.9997448
Done after 24 iterations.


## I-optimize ang mga parameter ng ansatz gamit ang MPS
Sa huli, io-optimize natin ang ansatz circuit para makagawa ito ng target state na may mas mataas na fidelity kaysa sa ating `comparison_fidelity`. Ang cost function na imi-minimize ay ang `MaximizeStateFidelity` at io-optimize gamit ang L-BFGS optimizer ng scipy.

In [7]:
final_circuit = aqc_ansatz.assign_parameters(aqc_final_parameters)
final_circuit.compose(subsequent_circuit, inplace=True)
final_circuit.draw("mpl", fold=-1)

<Image src="../docs/images/guides/qiskit-addons-aqc-get-started/extracted-outputs/45abbabe-0289-4a09-aa99-89f70bdc535d-0.svg" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Approximate quantum compilation for time evolution circuits](/docs/tutorials/approximate-quantum-compilation-for-time-evolution) tutorial.
</Admonition>